In [1]:
import pandas as pd
from sklearn.metrics import classification_report, accuracy_score, f1_score
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import time
import os
from imblearn.over_sampling import SMOTE

In [ ]:
# Constants
NUM_CLASSES = 2
RANDOM_STATE = 1082
MODEL_NAME = "LSTM Classifier (SMOTE)"


In [3]:
# Hyperparameters
INPUT_SIZE = 1
BATCH_SIZE = 64
HIDDEN_SIZE = 64
NUM_EPOCHS = 100
LEARNING_RATE = 0.001

In [ ]:
import sys as _sys
from pathlib import Path as _Path

_root = _Path.cwd().resolve()
while not (_root / "eval_metrics.py").is_file() and _root != _root.parent:
    _root = _root.parent
if str(_root) not in _sys.path:
    _sys.path.insert(0, str(_root))
from gps_ids import protocol


In [ ]:
EXP = protocol.smote_heldout("uav")
X_train_res = EXP.X_train
y_train_res = EXP.y_train
X_test_scaled = EXP.X_test
y_test = EXP.y_test

# Prepare data for LSTM: (samples, sequence_length, input_size)
X_train_np = X_train_res.values.astype('float32')
X_test_np = X_test_scaled.values.astype('float32')
y_train_np = y_train_res.astype('int64')
y_test_np = y_test.astype('int64')

SEQUENCE_LENGTH = X_train_np.shape[1]

X_train_lstm = X_train_np.reshape(-1, SEQUENCE_LENGTH, INPUT_SIZE)
X_test_lstm = X_test_np.reshape(-1, SEQUENCE_LENGTH, INPUT_SIZE)

In [6]:
# Convert to torch tensors
X_train_tensor = torch.tensor(X_train_lstm)
y_train_tensor = torch.tensor(y_train_np)
X_test_tensor = torch.tensor(X_test_lstm)
y_test_tensor = torch.tensor(y_test_np)

In [7]:
# Create DataLoader for batching
train_data = TensorDataset(X_train_tensor, y_train_tensor)
test_data = TensorDataset(X_test_tensor, y_test_tensor)
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_data, batch_size=BATCH_SIZE)

In [8]:
# Define the LSTM model
class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, num_classes):
        super(LSTMClassifier, self).__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True, num_layers=2)
        self.fc = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, _ = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

In [9]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed(RANDOM_STATE)
    torch.cuda.manual_seed_all(RANDOM_STATE)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [10]:
# initialize the model, loss function, and optimizer
model = LSTMClassifier(INPUT_SIZE, HIDDEN_SIZE, NUM_CLASSES).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [11]:
# Training loop
start_time = time.time()

for epoch in range(NUM_EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)

        # Calculate accuracy
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == y_batch).sum().item()
        total += y_batch.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    epoch_acc = correct / total
    print(f"Epoch {epoch+1}/{NUM_EPOCHS}, Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}")

train_time = time.time() - start_time

Epoch 1/100, Loss: 0.6671, Accuracy: 0.5697
Epoch 2/100, Loss: 0.2801, Accuracy: 0.9239
Epoch 3/100, Loss: 0.2432, Accuracy: 0.9359
Epoch 4/100, Loss: 0.1799, Accuracy: 0.9487
Epoch 5/100, Loss: 0.2097, Accuracy: 0.9351
Epoch 6/100, Loss: 0.1998, Accuracy: 0.9463
Epoch 7/100, Loss: 0.1985, Accuracy: 0.9463
Epoch 8/100, Loss: 0.1888, Accuracy: 0.9487
Epoch 9/100, Loss: 0.1734, Accuracy: 0.9519
Epoch 10/100, Loss: 0.1822, Accuracy: 0.9487
Epoch 11/100, Loss: 0.1942, Accuracy: 0.9471
Epoch 12/100, Loss: 0.1866, Accuracy: 0.9487
Epoch 13/100, Loss: 0.2172, Accuracy: 0.9439
Epoch 14/100, Loss: 0.2224, Accuracy: 0.9407
Epoch 15/100, Loss: 0.2160, Accuracy: 0.9407
Epoch 16/100, Loss: 0.6443, Accuracy: 0.5825
Epoch 17/100, Loss: 0.6961, Accuracy: 0.4800
Epoch 18/100, Loss: 0.7001, Accuracy: 0.5112
Epoch 19/100, Loss: 0.6941, Accuracy: 0.5152
Epoch 20/100, Loss: 0.6968, Accuracy: 0.4679
Epoch 21/100, Loss: 0.6929, Accuracy: 0.5016
Epoch 22/100, Loss: 0.6968, Accuracy: 0.4704
Epoch 23/100, Loss:

In [ ]:
# Evaluation
model.eval()
all_preds = []
all_probs = []
all_labels = []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        _, preds = torch.max(outputs, 1)
        all_probs.extend(torch.softmax(outputs, dim=1).cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.numpy())

In [13]:
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, digits=4))
accuracy = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='weighted', zero_division=0)
print("Accuracy:", accuracy * 100)
print("F1 Score:", f1 * 100)


Classification Report:
              precision    recall  f1-score   support

           0     0.8571    0.0480    0.0909      2500
           1     0.1371    0.9497    0.2395       398

    accuracy                         0.1718      2898
   macro avg     0.4971    0.4989    0.1652      2898
weighted avg     0.7582    0.1718    0.1113      2898

Accuracy: 17.184265010351968
F1 Score: 11.132199113683656


In [ ]:
from eval_metrics import evaluate, summary, time_inference_torch
from gps_ids.results import append_validated

_timing = time_inference_torch(model, next(iter(test_loader))[0].to(device))

row = evaluate(
    all_labels,
    all_preds,
    model_name=MODEL_NAME,
    family="SMOTE",
    y_proba=all_probs,
    train_time_sec=train_time,
    num_epochs=NUM_EPOCHS,
    parameters=list(dict(model.named_parameters()).keys()),
    timing=_timing,
    **EXP.eval_kwargs(),
)
append_validated(row, EXP.results_csv)
print(summary(row))
print(f"Results appended to {EXP.results_csv}")
